# S1 · Introduction to Generative AI, LLMs & Prompt Engineering
*A combined theory + hands-on notebook — follows the lecture deck slide-by-slide, with live demos where they fit.*

---

### Agenda (from the deck)
1. Introduction to Generative Models
2. Discriminative vs Generative Models
3. Large Language Models (LLMs)
4. Transformer Architecture refresher → Decoder-only (GPT)
5. Tokens & costing · Limitations of LLMs
6. Popular LLMs — GPT-4, LLaMA, Mistral, FLAN-T5, Falcon
7. **Introduction to Prompt Engineering** — crafting effective prompts
8. Zero-shot · Few-shot · Chain-of-Thought · modern techniques
9. LLM ecosystem — API-based vs self-hosted; latency, scalability, updates
10. Model evaluation
11. Use cases of Generative AI

> **Practical demos** are inserted at three points: **(A)** text generation & temperature, **(B)** next-token prediction, **(C)** prompt engineering (zero → few → chain-of-thought).
>
> **Retrieval-Augmented Generation (RAG) & LangChain are the topic of the *next* session** — noted here only where the deck first mentions them.

> **Running this:** `Runtime → Run all`. **CPU is fine** (free Colab). First run downloads two small models (~1.5 GB total). Prompt-engineering cells take ~10–40s each on CPU.

## Setup — install & import
Run this **first**. It installs a pinned Hugging Face `transformers` and imports what we need.

> **If you see `NameError: name 'torch' is not defined`** it means an incompatible
> `transformers` was already loaded. Just do **Kernel → Restart** (Colab: *Runtime → Restart
> session*), then **Run All**. The cell below pins a torch-compatible version and checks it for you.

In [5]:
# =============================================================================
# SETUP  -  RUN THIS CELL FIRST  (works on Google Colab and local machines)
# =============================================================================
# Why this looks fancy: the newest transformers (5.x) needs PyTorch >= 2.4. On an
# older torch it DISABLES PyTorch, and you then get the confusing error
# "NameError: name 'torch' is not defined". We pin transformers 4.46.3, which works
# with old AND new torch. This cell is SELF-HEALING: if an incompatible version is
# already loaded in the kernel, it reinstalls the right one and restarts the kernel
# for you (one time). If it restarts, just choose "Run All" again.
import sys, subprocess, importlib.metadata as _md

def _installed(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

# 1) Make sure the correct transformers is installed into THIS kernel's Python.
if not (_installed("transformers") or "").startswith("4.46"):
    print("Installing transformers==4.46.3 (compatible with your PyTorch)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.46.3", "accelerate"], check=True)

# 2) If a wrong version is already live in memory, a running kernel can't swap it.
#    Restart the kernel automatically - this is normal and happens at most once.
_live = sys.modules.get("transformers")
if _live is not None and not getattr(_live, "__version__", "").startswith("4.46"):
    print(f"An incompatible transformers ({_live.__version__}) is loaded in memory.")
    print(">>> Restarting the kernel now. When it comes back, choose 'Run All' again. <<<")
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        print("Could not auto-restart - please restart the kernel manually, then Run All.")
    raise SystemExit  # stop this run; continue after the restart

# 3) Safe to import now.
import warnings, textwrap
warnings.filterwarnings("ignore")

import torch, transformers
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()          # hide download / generation warnings

def show(text, width=90):
    """Pretty-print long model output so it wraps nicely."""
    print(textwrap.fill(str(text).strip(), width=width))

print(f"Setup OK - transformers {transformers.__version__}, torch {torch.__version__} - CPU ready.")

Setup OK - transformers 4.46.3, torch 2.11.0+cpu - CPU ready.


---
# 1 · Introduction to Generative AI

## What is Generative AI?
- **Generative AI**, also called **creative AI**, is a subset of machine learning that **generates novel / unique content** using artificial intelligence.
- Content types: **text, images, audio, video**, and more.
- It **differs from traditional (discriminative) AI**, which is designed to *recognise patterns and make predictions* from existing data.
- In natural language processing, generative models are typically **Large Language Models (LLMs)**.

## Discriminative AI vs Generative AI

| | **Discriminative AI** | **Generative AI** |
|---|------------------------|--------------------|
| **Goal** | Tell classes apart | Create new content |
| **Learns** | The *boundary* between classes | The *distribution* of the data |
| **Question** | "Is this review positive or negative?" | "Write a positive review." |
| **Output** | A label / probability | New text, image, audio… |
| **Examples** | Spam filter, sentiment classifier | GPT, DALL·E, Stable Diffusion |

## A brief history & the big picture
From early **n-gram** models → **RNNs / LSTMs** → the **Transformer (2017)** → GPT-style **LLMs** → the **ChatGPT moment (2022)** that brought Generative AI to everyone. Conceptually, generative models sit at the intersection of **deep learning** and **content creation**: instead of *labelling* data, they *produce* it.

**Demo A** below shows the defining trait — the model *creates new text* that we never wrote.

### Demo A — Generating content & the effect of *temperature*
We load a small GPT-2 (`distilgpt2`) and let it *continue* a sentence. **Temperature** controls randomness: **low → safe & repetitive**, **high → creative & surprising**.

In [6]:
# ------------------------------------------------------------------
# DEMO A: Generative AI creates new content; temperature controls it
# ------------------------------------------------------------------
generator = pipeline("text-generation", model="distilgpt2", device=-1)  # device=-1 => CPU

prompt = "Generative AI is a subset of machine learning that can create novel content such as"

for temp in [0.2, 0.7, 1.2]:
    print(f"\n===== temperature = {temp} =====")
    out = generator(prompt, max_new_tokens=45, do_sample=True, temperature=temp,
                    top_p=0.95, num_return_sequences=1,
                    pad_token_id=generator.tokenizer.eos_token_id)
    show(out[0]["generated_text"])


===== temperature = 0.2 =====
Generative AI is a subset of machine learning that can create novel content such as
stories, stories, and stories.

===== temperature = 0.7 =====
Generative AI is a subset of machine learning that can create novel content such as
articles.

===== temperature = 1.2 =====
Generative AI is a subset of machine learning that can create novel content such as
videos, books, stories, social media, search engines and social networking sites or web
service companies. While the general view of the internet is in an increasing consensus
with the Internet of Things, the idea of an AI capable of


**Observe:** the output is *grammatical* but often *not factual* — hold that thought (we name it **hallucination** in Section 5). Low temperature loops/repeats; high temperature invents. A **discriminative** model, by contrast, would only ever return a label.

---
# 2 · Large Language Models (LLMs)

## What is an LLM?
- Trained on **large amounts of raw text** in a **self-supervised** way.
- In **self-supervised learning**, the target is **computed automatically from the input** — *no manual labelling of data is required*.
- The model develops a **statistical understanding** of the language it was trained on.
- A raw LLM is **not very useful for a specific task** on its own. For specific tasks it goes through **transfer learning / fine-tuning** (supervised) for that application.
  - *Example:* **GPT** is the **LLM**; **ChatGPT** is the **application** built on top via fine-tuning.
- To the user, an LLM is an **interface between Deep Learning and Generative AI** — you talk to it in **natural-language prompts**, and it returns **conditional-probability-based** outputs.

## Why *"Large Language Model"*?
- **Large** — trained on huge datasets with **billions of trainable parameters**.
- **Language** — it deals with **text** (text in → text out).
- **Model** — it **predicts the next word / token**.

So: *an LLM is a neural network with billions of parameters, trained on large amounts of unlabelled text using self-supervised learning.* LLMs also show **emergent properties** such as **zero-shot learning**.

## Zero-shot learning
- The ability of a model to **complete a task it was never explicitly trained on**.
- LLMs get this "for free" from their **self-supervised** pre-training.
- *(We use this directly in Section 7 — zero-shot prompting.)*

---
# 3 · How do LLMs complete text? — The Transformer

LLMs complete text by **leveraging the Transformer architecture**.

## 3.1 Transformer refresher — key components
Transformers are deep-learning architectures designed for **sequential data**. Key parts:
- **Input Embeddings** — tokens turned into vectors
- **Positional Encoding** — injects word-order information
- **Multi-Head Self-Attention** — every token can look at every other token
- **Feed-Forward Neural Network** — per-token transformation
- **Residual Connections & Layer Normalization** — stabilise & speed up training

## 3.2 Why Transformers became popular
- **Parallel processing** (unlike sequential RNNs)
- Handles **long-range dependencies**
- **Faster training** than RNN / LSTM
- **Better contextual understanding**

## 3.3 Decoder-only Transformer (GPT)
GPT models use a **Decoder-Only Transformer**. How it works:
1. Input text is converted into **tokens**
2. Tokens are converted into **embeddings**
3. **Self-attention** identifies relationships between words
4. The model **predicts the next token probabilistically**
5. Output is generated **token-by-token**

**Key features:** **autoregressive** text generation · **context-aware** responses · trained via **self-supervised learning**.

## 3.4 Costing for LLM applications — Tokens
- A **token** is a segment of text — a word, sub-word, punctuation mark, or other meaningful unit — into which input is divided for processing.
- Models process text **as tokens, not full sentences**.
- Most LLM **APIs bill by the number of tokens** processed in **both input and output** → *tokens = cost*.

**Demo B** makes steps 3–5 concrete: we look at the model's **next-token probabilities**, then watch it generate **one token at a time**.

### Demo B — Next-token prediction (the heart of an LLM)
First we peek at the **probability the model assigns to candidate next tokens**; then we run the **autoregressive loop** by hand.

In [7]:
# ------------------------------------------------------------------
# DEMO B-1: the next-token PROBABILITY distribution
# ------------------------------------------------------------------
tok   = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

def top_next_tokens(text, k=5):
    inputs = tok(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits      # a score for every token, at every position
    next_logits = logits[0, -1, :]           # we only want the NEXT token (last position)
    probs = torch.softmax(next_logits, dim=-1)
    top = torch.topk(probs, k)
    print(f"Prompt: {text!r}")
    print(f"  Model's top-{k} guesses for the NEXT token:")
    for p, idx in zip(top.values, top.indices):
        print(f"    {tok.decode(idx)!r:12s} -> {p.item()*100:5.2f}%")
    print()

top_next_tokens("Once upon a")
top_next_tokens("The sky is")
top_next_tokens("Machine learning is a field of")

Prompt: 'Once upon a'
  Model's top-5 guesses for the NEXT token:
    ' time'      ->  7.02%
    ' second'    ->  6.51%
    ' moment'    ->  3.97%
    ' day'       ->  3.67%
    ' sudden'    ->  2.31%

Prompt: 'The sky is'
  Model's top-5 guesses for the NEXT token:
    ' the'       -> 10.03%
    ' falling'   ->  7.25%
    ' a'         ->  6.63%
    ' clear'     ->  3.71%
    ' full'      ->  2.86%

Prompt: 'Machine learning is a field of'
  Model's top-5 guesses for the NEXT token:
    ' research'  -> 13.07%
    ' study'     -> 10.30%
    ' thought'   ->  3.44%
    ' science'   ->  3.25%
    ' expertise' ->  2.51%



In [8]:
# ------------------------------------------------------------------
# DEMO B-2: autoregression - predict, append, repeat (greedy)
# ------------------------------------------------------------------
text = "Once upon a"
print(f"start: {text!r}")
for step in range(8):
    inputs = tok(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    next_id = torch.argmax(logits[0, -1, :])   # take the single most likely next token
    text += tok.decode(next_id)
    print(f"step {step+1}: {text!r}")

print("\nThis predict->append->repeat loop is exactly what happens inside .generate().")

start: 'Once upon a'
step 1: 'Once upon a time'
step 2: 'Once upon a time of'
step 3: 'Once upon a time of war'
step 4: 'Once upon a time of war,'
step 5: 'Once upon a time of war, the'
step 6: 'Once upon a time of war, the United'
step 7: 'Once upon a time of war, the United States'
step 8: 'Once upon a time of war, the United States was'

This predict->append->repeat loop is exactly what happens inside .generate().


**Takeaway:** the model outputs a **probability for every possible next token** — never one fixed answer. *Temperature* (Demo A) simply reshapes this distribution before a token is sampled. Every ChatGPT reply is built this way, one token at a time.

---
# 4 · Application building blocks & limitations

## 4.1 Generative-AI application building blocks
A typical LLM application combines: a **foundation model** + a **prompt** to steer it + optional **external knowledge** + a **framework** to wire it together + an **evaluation** step to check quality. *(We meet the knowledge/framework pieces — RAG & LangChain — in the **next session**.)*

## 4.2 Limitations & drawbacks of LLMs

| Limitation | What it means |
|------------|----------------|
| **Hallucinations** | Fluent, high-quality text that is **factually incorrect**. |
| **High compute time & cost** | Training needs huge data & compute; expensive, slow, and has **environmental impact**. |
| **Lack of transparency** | LLMs are a **black box** and lack explainability. |
| **Potential bias** | Output reflects **prejudice/bias** present in the training data. |
| **Data security risk** | Can handle/produce sensitive info → misuse for **phishing or deceptive spam**. |
| **Data poisoning** | Attackers manipulate training/fine-tuning data to plant **backdoors, biases or vulnerabilities**. |
| **Legality of content** | *Who owns AI-generated output?* is unsettled — copyright offices don't treat AI as an author, so raw output arguably belongs to the **public domain**. |

---
# 5 · The LLM landscape — popular models

**Scale keeps growing:** the "chronology" and "magnitude" slides show model sizes climbing into the tens/hundreds of **billions of parameters**, trained on **trillions of tokens**. More parameters + more training tokens generally → more capable (and more expensive) models.

## A few popular LLMs

| Model | Maker | Open? | Notes |
|-------|-------|-------|-------|
| **GPT-3 / GPT-4** | OpenAI | Closed (API) | *Generative Pre-trained Transformer*; inputs are called **prompts**; build apps via the OpenAI **API**. GPT-3 trained on ~0.5T tokens. |
| **FLAN-T5** | Google Research | Open | *Fine-tuned Language Net* + *Text-To-Text Transfer Transformer* — an **encoder-decoder** model pre-trained on a multi-task mix; fast & efficient; good at translation, classification, QA. Comes in small→XXL checkpoints. |
| **LLaMA 2** | Meta | Open | 7B / 13B / 70B variants (bigger = more powerful); chat-tuned versions suit conversational / prompt-engineering apps; trained on **2T tokens**; strong open alternative to GPT-3.5. Can be **quantized** (4/5-bit) to run on modest GPUs. |
| **Falcon** | TII (UAE) | Open | Causal **decoder-only**; sizes up to **180B** params, trained on **3.5T tokens**; transparent, open source, rich training data. |
| **Mistral / Mixtral** | Mistral AI | Open-weight | *Mistral 7B*, *Mixtral 8×7B*; **high performance at lower compute**, efficient inference, enterprise-friendly and **self-hostable**. |

> **GPT recap:** *Generative Pre-trained Transformer* — an OpenAI LLM that returns text outputs for text inputs (**prompts**); you can build your own app on the **GPT API**.

---
# 6 · Introduction to Prompt Engineering  ⭐ *(main focus)*

## 6.1 What is prompt engineering?
- The process of **designing and crafting prompts** for conversational / generative AI models to get **as close as possible to the exact output desired**.
- In plain terms: it makes sure the model **understands the question** and gives the **best answer**.
- The model's **parameters are NOT changed** — you use the LLM as-is, either via:
  1. a **web interface** (e.g. ChatGPT), or
  2. **programmatically / API** (e.g. OpenAI API, Hugging Face libraries).
- **Drawbacks:** large model sizes; great for **generic** tasks but weaker on **domain-specific** ones.

## 6.2 Prompt structure — three components
Prompts to LLM APIs generally have three structural parts:
1. **System / instruction** — *who* the model is and the rules ("You are a helpful data scientist…").
2. **Context** — background info or examples the model should use.
3. **Query** — the actual question / task.

## 6.3 Prompt parameters — four knobs
Beyond structure, four parameters shape the response. The two key ones are **Temperature** and **Output length (max tokens)**.
- **Temperature** — randomness/creativity (low = focused, high = creative).
- **Top-P** (nucleus sampling) — alternative randomness control. **Temperature and Top-P are not tuned together.**
- **Output length / max tokens** — caps length (and cost).
- **Stop sequences** — where generation should halt.

## 6.4 Prompting is *iterative*
Effective prompting is a loop: **draft → test → inspect the output → refine**. You rarely get the perfect prompt on the first try.

Now we switch to a small **instruction-tuned** chat model so prompts are actually *followed* (recall Section 2: raw GPT babbles → fine-tuning turns it into a useful assistant like ChatGPT).

In [10]:
# ------------------------------------------------------------------
# Load a small INSTRUCTION-TUNED chat model + a helper to prompt it
# ------------------------------------------------------------------
INSTRUCT_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # small, chat-tuned, CPU-friendly

chat_tok   = AutoTokenizer.from_pretrained(INSTRUCT_ID)
chat_model = AutoModelForCausalLM.from_pretrained(INSTRUCT_ID, torch_dtype="auto")
chat_model.eval()

def ask(user, system="You are a helpful assistant.", max_new_tokens=200, temperature=0.0):
    """Send system + user messages via the chat template; return the model's reply.
    temperature=0 => deterministic (ideal for classification)."""
    messages = [{"role": "system", "content": system},
                {"role": "user",   "content": user}]
    prompt = chat_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = chat_tok(prompt, return_tensors="pt")
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=chat_tok.eos_token_id)
    if temperature and temperature > 0:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        kwargs.update(do_sample=False)
    with torch.no_grad():
        out = chat_model.generate(**inputs, **kwargs)
    return chat_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Instruction-tuned model loaded. (CPU generation ~10-40s per call.)")
show(ask("In one sentence, what is prompt engineering?"))

Instruction-tuned model loaded. (CPU generation ~10-40s per call.)
Prompt engineering involves creating and managing prompts for AI systems to generate
natural language text or images based on user input.


## 6.5 Prompting approaches
- **Zero-shot prompting** — the model is given a task with **no examples**; it relies purely on its **pre-trained knowledge**.
- **One-shot prompting** — the model is given **a single example** and must generalise from it.
- **Few-shot prompting** — the model is given **a few input→output examples** so it learns the **structure/format** of the task and adapts. *(Note: few-shot does **not** fine-tune the model — it just leverages pre-trained knowledge with minimal examples.)*

### Demo C-1 — Zero-shot prompting

In [11]:
# ------------------------------------------------------------------
# DEMO C-1: ZERO-SHOT (no examples - rely on pre-trained knowledge)
# ------------------------------------------------------------------
review = "The lecture on Transformer architecture was insightful, but a bit fast-paced."

zero = f"""Classify the sentiment of the review as Positive, Negative, or Neutral.
Answer with only one word.

Review: "{review}"
Sentiment:"""
print("ZERO-SHOT prompt:\n", zero, "\n")
print("Model:", ask(zero))

ZERO-SHOT prompt:
 Classify the sentiment of the review as Positive, Negative, or Neutral.
Answer with only one word.

Review: "The lecture on Transformer architecture was insightful, but a bit fast-paced."
Sentiment: 

Model: Neutral


### Demo C-2 — Few-shot prompting
We now prepend a few **worked examples** so the model copies the pattern.

In [12]:
# ------------------------------------------------------------------
# DEMO C-2: FEW-SHOT (show the pattern with examples)
# ------------------------------------------------------------------
few = f"""Classify the sentiment as Positive, Negative, or Neutral.

Review: "I loved the session"          -> Positive
Review: "Too much theory, I was lost"  -> Negative
Review: "It was an okay session"       -> Neutral

Review: "{review}"                     ->"""
print("FEW-SHOT prompt:\n", few, "\n")
print("Model:", ask(few))

FEW-SHOT prompt:
 Classify the sentiment as Positive, Negative, or Neutral.

Review: "I loved the session"          -> Positive
Review: "Too much theory, I was lost"  -> Negative
Review: "It was an okay session"       -> Neutral

Review: "The lecture on Transformer architecture was insightful, but a bit fast-paced."                     -> 

Model: Positive


### Demo C-3 — Chain-of-Thought (CoT) prompting
For multi-step problems, asking the model to **"think step by step"** before answering improves reasoning. This is the exact token-count problem from the deck's demo.

> **Watch for it:** our tiny 0.5B model lays out neat steps but still gets the **arithmetic wrong** (correct answer ≈ **750 million**), and the two variants may even disagree. That's a perfect live example of **hallucination** (§4.2): CoT improves the *structure* of reasoning, but a small model still can't be trusted on numbers.

In [13]:
# ------------------------------------------------------------------
# DEMO C-3: CHAIN-OF-THOUGHT - reason step by step
# ------------------------------------------------------------------
problem = ("A company trained an LLM on 1.5 trillion tokens. "
           "Each training step processes 2,000 tokens. "
           "Approximately how many steps were needed?")

print("A) DIRECT (no reasoning shown):")
show(ask(f"{problem}\nAnswer in one short sentence with the final number only.", max_new_tokens=160))

print("\nB) CHAIN-OF-THOUGHT (reason step by step, then answer):")
show(ask(f"{problem}\nLet's think step by step, then state the final answer.", max_new_tokens=420))

# Correct answer: 1,500,000,000,000 / 2,000 = 750,000,000  (750 million) steps.
# Watch: this small model is CONFIDENTLY WRONG (and the two variants may even disagree!)
# -> a live demonstration of HALLUCINATION and why a small LLM can't be trusted on arithmetic.

A) DIRECT (no reasoning shown):
The answer is approximately 75 million steps.

B) CHAIN-OF-THOUGHT (reason step by step, then answer):
To determine the number of training steps required for the LLM to process 1.5 trillion
tokens, we can follow these steps:  1. **Convert the total number of tokens into units:**
- We know that \(1\) trillion tokens is equal to \(1,000,000,000,000\) tokens.    -
Therefore, \(1.5\) trillion tokens is:      \[      1.5 \times 1,000,000,000,000 =
1,500,000,000,000 \text{ tokens}      \]  2. **Determine the processing rate per step:**
- The LLM processes \(2,000\) tokens per step.  3. **Calculate the number of steps
needed:**    - To find the number of steps required to process \(1,500,000,000,000\)
tokens at a rate of \(2,000\) tokens per step, we divide the total number of tokens by the
processing rate:      \[      \frac{1,500,000,000,000}{2,000} = 75,000,000,000 \text{
steps}      \]  Therefore, approximately \(75,000,000,000\) steps were needed to train 

## 6.6 Modern prompt-engineering techniques

| Technique | Idea |
|-----------|------|
| **Role prompting** | Assign the model a role — *"Act as a Data Scientist."* |
| **System prompting** | Define the model's **behaviour and constraints**. |
| **Structured prompting** | Ask for output as **tables / JSON / bullets**. |
| **Prompt chaining** | Break a complex task into **smaller sequential prompts**, each output feeding the next. |
| **Retrieval-Augmented Generation (RAG)** | Combine **external knowledge retrieval** with the LLM. *Covered in the **next session** (with LangChain).* |

### Demo C-4 — Role / System prompting + Structured (JSON) output

In [14]:
# ------------------------------------------------------------------
# DEMO C-4: ROLE / SYSTEM prompting  +  STRUCTURED (JSON) output
# ------------------------------------------------------------------
print("ROLE + SYSTEM PROMPTING (defined persona):")
show(ask(
    "Explain what a 'token' is in an LLM.",
    system="You are a senior data scientist teaching M.Tech students. "
           "Be concise and include exactly one everyday analogy.",
    max_new_tokens=160,
))

print("\nSTRUCTURED OUTPUT (force machine-readable JSON):")
show(ask(
    'Extract the model, developer, and parameter sizes as JSON with keys '
    '"model", "developer", "parameters".\n'
    'Text: "LLaMA 2 is an open-source LLM from Meta, available in 7B, 13B and 70B sizes."',
    max_new_tokens=150,
))

ROLE + SYSTEM PROMPTING (defined persona):
In natural language processing (NLP), a token refers to the smallest unit of text that can
be processed by a machine learning model. It's like a building block for larger words or
phrases, which together form sentences. Tokens are crucial because they represent
individual units of information rather than whole sentences. For example, "the quick brown
fox" would be represented as three tokens: "the", "quick", and "brown". This allows models
to understand context and relationships between different parts of speech, making them
more versatile and effective at tasks such as sentiment analysis, named entity
recognition, and language understanding.

STRUCTURED OUTPUT (force machine-readable JSON):
```json {   "model": "LLaMA 2",   "developer": "Meta",   "parameters": {     "size": [
{         "name": "B",         "value": "7B"       },       {         "name": "M",
"value": "13B"       },       {         "name": "S",         "value": "70B"       }   

## 6.7 Prompt chaining & classification tasks
- **Prompt chaining** breaks a complex task into a series of steps, where **the output of one step is the input to the next** (e.g. *extract facts → summarise them → translate the summary*).
- **Classification via prompting:** the sentiment demos above are exactly this — the LLM assigns a **label** from a fixed set, steered entirely by the prompt (zero- or few-shot), with **no retraining**.

---
# 7 · Overview of the LLM ecosystem

## Accessing LLMs
- **Open-source / open-weight** LLMs (LLaMA, Mistral, Falcon, FLAN-T5) can be **self-hosted** on your own servers.
- **Closed** models (e.g. GPT-4) are reached through a **managed API**.

## API-based vs Self-hosted deployment

| | **API-based** (e.g. GPT API) | **Self-hosted** (e.g. LLaMA / Mistral) |
|---|------------------------------|-----------------------------------------|
| **Setup** | Easy; managed infrastructure | Full control; requires **GPU infrastructure** |
| **Cost** | Subscription / **per-token** pricing | Infrastructure purchase + **maintenance** |
| **Integration** | Faster integration | More engineering effort |
| **Privacy / security** | Data leaves your walls | **Better privacy / security** |

## Latency, scalability & update strategies
- **Latency:** APIs add network round-trips; self-hosting can be lower-latency at scale.
- **Scalability:** APIs scale effortlessly (someone else's problem); self-hosting means you own scaling.
- **Updates:** with an API the **provider updates the model** for you (behaviour can change under you); self-hosting means **you choose when/whether to update** — more stable, more work.

---
# 8 · Model evaluation

Evaluating LLMs is **hard — often more art than science** (true even for GPT-4-class models).

## 8.1 Benchmarks & leaderboards
- Evaluate against a **benchmark dataset** (e.g. the Open LLM datasets).
- **Open LLM Leaderboard** — a public benchmark by **Hugging Face**, updated regularly.
- Four common benchmark datasets: **1) ARC · 2) HellaSwag · 3) MMLU · 4) TruthfulQA**.
- **High scores do *not* necessarily mean a good model** for *your* task.
- For **open-ended tasks**, also use: **human evaluation**, **NLP metrics (BLEU / ROUGE)**, or an **auxiliary fine-tuned LLM**.

## 8.2 ROUGE  *(Recall-Oriented Understudy for Gisting Evaluation)*
- **What:** how closely the words in a generated summary **overlap** with a reference summary. Score **0–1** (0 = no overlap, 1 = perfect).
- **How:** compares overlapping word sequences — pairs (**bigrams**), triples (**trigrams**), etc.
- **When:** best for **extractive** text-generation tasks.

## 8.3 BERTScore
- **What:** whether the output conveys the **same meaning** as the reference, **regardless of exact words**. Score **0–1**.
- **How:** compares the **contextual meaning** via **embeddings**, not exact word overlap.
- **When:** ideal for **abstractive** generation tasks.

## 8.4 LLM-as-a-Judge
- **What:** quality of the output against defined criteria (e.g. conciseness, factual accuracy).
- **How:** a **powerful LLM (e.g. GPT-4)** is prompted to score/rate the output on the given metrics.
- **When:** ideal for **abstractive** text-generation tasks.

## 8.5 SUMMEVAL
- An **evaluation framework** for LLM outputs, especially **summarization**, scoring across several key dimensions — a common source of **LLM-as-a-Judge** metrics.

## 8.6 Worked example — how ROUGE is actually computed
Take a tiny **reference** and **candidate** summary:
- **Reference:**  `the cat sat on the mat`
- **Candidate:** `the cat sat on mat`

**ROUGE-1** counts overlapping **single words (unigrams)**:

| | words (count) | matched |
|---|---|---|
| Reference | the, cat, sat, on, the, mat (6) | — |
| Candidate | the, cat, sat, on, mat (5) | the, cat, sat, on, mat → **5** |

- **Recall** = matches ÷ reference words = 5 ÷ 6 = **0.83**
- **Precision** = matches ÷ candidate words = 5 ÷ 5 = **1.00**
- **F1** = 2·P·R ÷ (P + R) = (2 × 1.00 × 0.83) ÷ 1.83 = **0.91**

**ROUGE-2** does the same with **word pairs (bigrams)**:
- Reference bigrams: `the cat`, `cat sat`, `sat on`, `on the`, `the mat` (5)
- Candidate bigrams: `the cat`, `cat sat`, `sat on`, `on mat` (4)
- Overlap: `the cat`, `cat sat`, `sat on` → **3** → Recall = 3÷5 = 0.60, Precision = 3÷4 = 0.75, **F1 = 0.67**

**ROUGE-L** uses the **Longest Common Subsequence** (`the cat sat on … mat`) instead of fixed n-grams, so it rewards in-order overlap even when words aren't consecutive.

> **Built-in limitation:** ROUGE only sees **words**. A perfect paraphrase — `a feline rested on the rug` — shares almost no words with the reference, so ROUGE scores it **low** even though the meaning is identical. That is exactly why we need meaning-based metrics.

## 8.7 How BERTScore captures *meaning*
Instead of matching exact words, BERTScore works in three steps:
1. Run both texts through a Transformer (e.g. BERT) to get a **contextual embedding for every token**.
2. **Match each candidate token to the most similar reference token** using **cosine similarity** — so `film` ↔ `movie` scores high even though the words differ.
3. Aggregate those similarities into **Precision, Recall, and F1** (0–1).

Because `film` and `movie` have nearly identical embeddings, BERTScore **rewards the paraphrase that ROUGE penalised** — this is the key difference between the two metrics.

> **Caveat:** BERTScore has a **high baseline** — even unrelated sentences often score ~0.7 — so interpret scores **relatively** (compare candidates to each other), not against 0.

**Rule of thumb:** ROUGE → **extractive** tasks (overlap matters) · BERTScore → **abstractive** tasks (meaning matters).

## 8.8 LLM-as-a-Judge — an example prompt
You hand the output to a **strong LLM** together with a clear rubric, and let it grade. A typical judge prompt looks like:

```
System:  You are a strict, impartial evaluator.
User:    Score the SUMMARY for factual accuracy on a 1–5 scale
         (1 = many errors, 5 = fully faithful to the SOURCE).
         Reply as ->  Score: <n> — <one-line reason>.

         SOURCE:  "Self-attention lets every token attend to every other token…"
         SUMMARY: "Self-attention makes the model slower and is only used in image models."
```

A capable judge (e.g. **GPT-4**) would return something like:
`Score: 1 — the summary states false facts about self-attention.`

- **Strengths:** flexible; handles open-ended quality that ROUGE/BERTScore can't (tone, reasoning, helpfulness).
- **Risks:** the judge can be **biased or simply wrong**, it **costs tokens**, and it is only as reliable as the model doing the judging — which is why a *powerful* model is used as the judge.

---
# 9 · Building LLM applications (brief) → *bridge to the next session*

- A production LLM system (see the **Databricks** reference architecture in the deck) chains together a foundation model, prompt templates, **external knowledge retrieval**, orchestration, and evaluation.
- **Service providers / tools:** **LangChain** (open-source hub for LLM apps), **Fixie** (AI-agent platform), and cloud offerings like **Microsoft Semantic Kernel** and **Google Vertex AI**.

> **Next session:** we build a real **Retrieval-Augmented Generation (RAG)** pipeline with **LangChain** — *Load → Split → Embed → Store → Retrieve → Generate* — to ground answers in your own documents and cut hallucination. That's why RAG/LangChain are only *named* here, not demoed.

---
# 10 · Use cases of Generative AI

- **Creative industries** — art, music, fashion, content generation, music composition.
- **Medical** — drug discovery, medical imaging (analysing X-rays / MRIs), new insights into diseases → faster, more accurate diagnosis and timely intervention.
- **Personalized marketing** — tailored content for targeted campaigns.
- **Architectural rendering** — architectural design, urban planning, landscape generation.
- **Scientific discovery** — simulations and synthetic data generation for research.
- **Recommender systems** — personalized content and product recommendations.

**Business problems solved by GenAI** span **Retail** (product descriptions, support bots, personalization), **Healthcare** (imaging, documentation, discovery), and **Tech** (code generation, search, assistants).

---
# Recap & exercises

**The thread of today:** *Generative AI* creates new content → for text that means **LLMs**, which are giant **next-token predictors** built on the **decoder-only Transformer**, trained **self-supervised** → a raw model babbles until **instruction-tuning + prompt engineering** (zero-shot, few-shot, chain-of-thought, roles, structured output) make it useful → and we must weigh **limitations, deployment (API vs self-hosted), and evaluation** before trusting it.

### Try it yourself
1. **Demo A:** find a temperature where `distilgpt2` turns to nonsense — what's the trade-off?
2. **Demo B:** feed a new prompt to `top_next_tokens` — do the top guesses match your intuition?
3. **Demo C:** write a *few-shot* prompt that classifies text into **4 custom categories** of your choice.
4. **Demo C-3:** try another multi-step word problem — does chain-of-thought help or hurt?
5. **Design:** to answer questions over *your college's PDFs*, would you pick an **API** or **self-hosted** model, and why (Section 7)?

> **Next session → Retrieval-Augmented Generation (RAG) with LangChain.**

### Thank you